# Python APIs — IBM Watson Speech to Text & Language Translator

**What you will build:**  
A pipeline that (1) converts an audio lecture to text using Watson Speech to Text, then (2) translates that text into Spanish and French using Watson Language Translator.

**Sections:**
1. [APIs through pandas](#part1) — understand the concept locally first  
2. [REST API fundamentals](#part2) — how APIs work over the internet  
3. [Watson Speech to Text](#part3) — transcribe an MP3 file  
4. [Watson Language Translator](#part4) — translate the transcript  

> **Before you start:** copy `.env.example` to `.env` and fill in your IBM Watson API keys.  
> See the README for step-by-step instructions on getting free API keys.

## Setup

Run this cell once to install all required packages.  
If you are using a virtual environment, make sure it is activated first.

In [ ]:
!pip install ibm-watson ibm-cloud-sdk-core pandas wget python-dotenv --quiet

### Load environment variables

We use `python-dotenv` to read API keys from a local `.env` file so they never appear in the notebook source code.

```
SPEECH_TO_TEXT_API_KEY=your_key
SPEECH_TO_TEXT_URL=https://api.us-south.speech-to-text.watson.cloud.ibm.com
LANGUAGE_TRANSLATOR_API_KEY=your_key
LANGUAGE_TRANSLATOR_URL=https://api.us-south.language-translator.watson.cloud.ibm.com
```

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env into os.environ

S2T_API_KEY = os.getenv("SPEECH_TO_TEXT_API_KEY")
S2T_URL     = os.getenv("SPEECH_TO_TEXT_URL")
LT_API_KEY  = os.getenv("LANGUAGE_TRANSLATOR_API_KEY")
LT_URL      = os.getenv("LANGUAGE_TRANSLATOR_URL")

missing = [name for name, val in [
    ("SPEECH_TO_TEXT_API_KEY", S2T_API_KEY),
    ("SPEECH_TO_TEXT_URL",     S2T_URL),
    ("LANGUAGE_TRANSLATOR_API_KEY", LT_API_KEY),
    ("LANGUAGE_TRANSLATOR_URL",     LT_URL),
] if not val or val.strip() == ""]

if missing:
    print("WARNING: the following variables are not set in your .env file:")
    for m in missing:
        print(f"  - {m}")
    print("\nSee README.md → 'Getting IBM Watson API Keys' for instructions.")
else:
    print("All credentials loaded successfully.")

<a id='part1'></a>
---
## Part 1 — APIs Through pandas

An **API (Application Programming Interface)** is a contract between two pieces of software that defines:  
- what inputs you can send  
- what outputs you will receive  
- how errors are reported  

You have already been using APIs without necessarily calling them that.  
Every time you call `df.mean()` on a pandas DataFrame, you are using the pandas API:  
you provide data, the interface processes it, and you receive a result back.

The location of the other software — on your machine or on a remote server — is an implementation detail.  
The **concept** is the same in both cases.

In [ ]:
import pandas as pd

data = {
    'Score_A': [2, 4, 6, 4],
    'Score_B': [12, 4, 6, 8],
    'Score_C': [2, 5, 7, 8],
}

df = pd.DataFrame(data)
print("DataFrame:")
print(df)

# df.mean() is an API call:
#   Input  → the numbers in each column
#   Output → the average of each column
print("\nColumn means (via the pandas API):")
print(df.mean())

<a id='part2'></a>
---
## Part 2 — REST API Fundamentals

**REST (Representational State Transfer)** is the most common style for web APIs.  
A REST API exposes *resources* (data or operations) at *URLs*, and you interact with them using **HTTP methods**:

| HTTP Method | Meaning | Example |
|---|---|---|
| `GET`    | Retrieve data              | Fetch a list of supported languages |
| `POST`   | Send data to be processed  | Upload an audio file for transcription |
| `PUT`    | Replace an existing record | Update a stored document |
| `DELETE` | Remove a record            | Delete a custom translation model |

Every interaction follows a **request → response** cycle:

```
Client                              Server
  │                                   │
  │── POST /recognize (audio bytes) ──▶│
  │                                   │  process audio
  │◀─────── 200 OK (JSON transcript) ──│
```

**Authentication** is usually handled by including a secret **API key** in the request headers.  
The server uses the key to verify your identity and track usage (important for billing on paid plans).

<a id='part3'></a>
---
## Part 3 — Watson Speech to Text

We will use IBM Watson's Speech to Text service to transcribe an audio lecture file.

### How the IBM Watson SDK works

The `ibm-watson` Python SDK wraps the raw HTTP calls for you:
1. You create an `IAMAuthenticator` with your API key.
2. The SDK exchanges the key for a short-lived bearer token internally.
3. Every subsequent SDK method call includes that token in the `Authorization` header automatically.

You never have to write raw `requests.post(...)` calls yourself.

In [ ]:
from ibm_watson import SpeechToTextV1
from ibm_cloud_sdk_core.authenticators import IAMAuthenticator

authenticator_s2t = IAMAuthenticator(S2T_API_KEY)
speech_to_text = SpeechToTextV1(authenticator=authenticator_s2t)
speech_to_text.set_service_url(S2T_URL)

print("Speech to Text client ready:", speech_to_text)

### Download the sample audio file

We download a short MP3 lecture about Polynomial Regression from IBM's public object storage.  
This represents the audio you would record or receive in a real-world application.

In [ ]:
import wget

audio_filename = "PolynomialRegressionandPipelines.mp3"
audio_url = (
    "https://s3-api.us-geo.objectstorage.softlayer.net/"
    "cf-courses-data/CognitiveClass/PY0101EN/labs/"
    "PolynomialRegressionandPipelines.mp3"
)

if not os.path.exists(audio_filename):
    wget.download(audio_url, audio_filename)
    print(f"\nDownloaded: {audio_filename}")
else:
    print(f"File already exists: {audio_filename}")

### Transcribe the audio

We open the file in **binary read mode** (`"rb"`) because we are sending raw bytes over HTTP, not a text string.  
The `content_type` parameter tells the API what audio format to expect.

In [ ]:
with open(audio_filename, mode="rb") as audio_file:
    response = speech_to_text.recognize(
        audio=audio_file,
        content_type="audio/mp3"
    )

# The SDK returns a DetailedResponse; .get_result() gives us the raw dict
transcript_data = response.get_result()
print("Raw JSON response (first result):")
print(transcript_data["results"][0])

In [ ]:
# Flatten the nested JSON into a readable DataFrame
from pandas import json_normalize

results_df = json_normalize(transcript_data["results"], "alternatives")
print("Transcription results:")
results_df

In [ ]:
# Extract the highest-confidence transcript
recognized_text = transcript_data["results"][0]["alternatives"][0]["transcript"]
print("Transcript:")
print(recognized_text)

<a id='part4'></a>
---
## Part 4 — Watson Language Translator

Now we pass the English transcript to Watson Language Translator.  
The service accepts a `model_id` string in the form `source-target` (e.g. `en-es` = English to Spanish).

### Initialization (same pattern as Speech to Text)

In [ ]:
from ibm_watson import LanguageTranslatorV3

authenticator_lt = IAMAuthenticator(LT_API_KEY)
language_translator = LanguageTranslatorV3(
    version="2018-05-01",
    authenticator=authenticator_lt
)
language_translator.set_service_url(LT_URL)

print("Language Translator client ready:", language_translator)

### Explore available language pairs

The service supports ~70 languages. Let's list them all so you can see what is possible.

In [ ]:
languages_result = language_translator.list_identifiable_languages().get_result()
languages_df = json_normalize(languages_result, "languages")
print(f"Total identifiable languages: {len(languages_df)}")
languages_df.head(10)

### Translate: English → Spanish

In [ ]:
es_response = language_translator.translate(
    text=recognized_text,
    model_id="en-es"
).get_result()

spanish_translation = es_response["translations"][0]["translation"]
print("Spanish:")
print(spanish_translation)

### Translate: Spanish → English

Round-tripping (English → Spanish → English) is a useful sanity check:  
if the back-translation closely matches the original, the translation quality is high.

In [ ]:
en_response = language_translator.translate(
    text=spanish_translation,
    model_id="es-en"
).get_result()

back_to_english = en_response["translations"][0]["translation"]
print("Back to English:")
print(back_to_english)

### Translate: English → French

In [ ]:
fr_response = language_translator.translate(
    text=recognized_text,
    model_id="en-fr"
).get_result()

french_translation = fr_response["translations"][0]["translation"]
print("French:")
print(french_translation)

---
## Summary

| Step | API Used | Input | Output |
|---|---|---|---|
| 1 | Watson Speech to Text | MP3 audio file | English transcript |
| 2 | Watson Language Translator (`en-es`) | English text | Spanish text |
| 3 | Watson Language Translator (`es-en`) | Spanish text | English text (round-trip check) |
| 4 | Watson Language Translator (`en-fr`) | English text | French text |

**What you practiced:**
- Using `IAMAuthenticator` to securely connect to IBM Cloud services
- Making POST requests through an SDK (no raw `requests` code needed)
- Parsing nested JSON responses with `json_normalize`
- Loading API credentials from environment variables (the `.env` pattern)

**Next steps to explore:**
- Try a different `model_id` (see the `languages_df` table above for available language codes)
- Record your own voice and transcribe it
- Build a simple Flask or FastAPI web app that exposes this pipeline as your own REST endpoint